# Medical Prompt Analysis with LIME and Llama Med42

Testing LIME explanations with the Llama3-med42-8b model.

In [ ]:
import sys
import os
import requests
import json
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lime.lime_text import LimeTextExplainer
from utils.medical_processor import MedicalTermProcessor
from IPython.display import HTML, display

In [2]:
# Model configuration
MODEL_CONFIG = {
    "url": "http://localhost:11434/api/chat",
    "model": "llama3-med42-8b",
    "system_prompt": "You are a medical expert assistant analyzing clinical information."
}

# Initialize LIME explainer
explainer = LimeTextExplainer(
    class_names=['relevant', 'not relevant'],
    split_expression='\W+',
    random_state=42
)

# Initialize medical processor
medical_processor = MedicalTermProcessor()

def get_model_response(text, stream=False):
    """Get response from Llama model"""
    try:
        payload = {
            "model": MODEL_CONFIG["model"],
            "messages": [
                {"role": "system", "content": MODEL_CONFIG["system_prompt"]},
                {"role": "user", "content": text}
            ],
            "stream": stream
        }
        
        response = requests.post(MODEL_CONFIG["url"], json=payload)
        if response.status_code == 200:
            data = response.json()
            return data.get("message", {}).get("content", "")
        return f"Error: {response.status_code}"
    except Exception as e:
        return f"Error: {str(e)}"

# Test model connection
test_response = get_model_response("Test connection")
print("Model initialized and tested.")

Loaded 6762 medical terms
Model initialized and tested.


In [3]:
def predictor(texts):
    """Prediction function for LIME"""
    predictions = []
    for text in texts:
        response = get_model_response(text)
        
        # Get medical terms from input and response
        input_terms = set(text.lower().split())
        response_terms = set(response.lower().split())
        medical_terms = medical_processor.medical_terms
        
        # Calculate medical relevance
        medical_overlap = len((input_terms | response_terms) & medical_terms)
        relevance = min(0.5 + (medical_overlap * 0.1), 0.9)
        
        predictions.append([1 - relevance, relevance])
    
    return np.array(predictions)

# Test predictor
test_text = "Patient has severe headache"
pred = predictor([test_text])
print(f"Test prediction shape: {pred.shape}")

Test prediction shape: (1, 2)


In [ ]:
def explain_prompt(text, num_features=10):
    """Generate and visualize explanation for a prompt"""
    print(f"Analyzing: {text}")
    
    # Get model response
    response = get_model_response(text)
    print(f"\nModel response: {response}")
    
    # Generate LIME explanation
    exp = explainer.explain_instance(
        text,
        predictor,
        num_features=num_features,
        num_samples=100
    )
    
    # Visualize explanation as HTML
    html = exp.as_html()
    display(HTML(html))
    
    # Visualize explanation as bar chart
    weights = exp.as_list()
    feat_df = pd.DataFrame(weights, columns=['Feature', 'Weight'])
    
    plt.figure(figsize=(10, 6))
    colors = ('red' if w > 0 else 'blue' for w in feat_df['Weight'])
    plt.barh(range(len(weights)), feat_df['Weight'], color=colors)
    plt.yticks(range(len(weights)), feat_df['Feature'])
    plt.xlabel('Impact')
    plt.title('Feature Importance')
    plt.tight_layout()
    plt.show()
    
    return feat_df, response

In [ ]:
# Test cases
test_prompts = [
    "I have a headache",
]

# Analyze first test case
results, response = explain_prompt(test_prompts[0])
print("\nFeature importance:")
print(results)

Analyzing: Patient presents with fever of 39°C and persistent cough

Model response: As a biomedical expert, my response would be: Based on the given symptoms - fever of 39°C and persistent cough, it is likely that this patient may have pneumonia or bronchitis. Further examination such as chest X-ray, blood tests, and possibly a sputum culture are required to confirm diagnosis and determine appropriate treatment options.

For pneumonia, treatments could include antibiotics (such as amoxicillin-clavulanate) depending on the bacterial cause identified, and supportive care like antipyretics for fever management. For bronchitis, mainly viral, symptomatic relief with over-the-counter cough suppressants or expectorants may be recommended along with hydration.

It's important to note that this is a preliminary assessment based solely on provided symptoms and could change upon further investigation.


KeyboardInterrupt: 

## Interactive Testing

Test your own medical prompts below:

In [ ]:
custom_prompt = "I have a headache"
results, response = explain_prompt(custom_prompt)

## Comparison with SHAP

LIME provides a different perspective on feature importance compared to SHAP:
- LIME focuses on local interpretability by creating a simpler, interpretable model around each prediction
- SHAP provides a more global view of feature importance based on game theory

Key differences in the results:
1. LIME shows how words work together in context
2. SHAP shows individual word contributions
3. LIME's explanations are more focused on the local decision boundary